<h3 style="text-align: center;"><b>Школа глубокого обучения ФПМИ МФТИ</b></h3>

<h3 style="text-align: center;"><b>Домашнее задание. Детекция объектов</b></h3>

В этом домашнем задании мы продолжим работу над детектором из семинара, поэтому при необходимости можете заимствовать оттуда любой код.

Домашнее задание можно разделить на следующие части:

* Переделываем модель [4]
  * Backbone[1],
  * Neck [2],
  * Head [1]
* Label assignment [3]:
  * TAL [3]
* Лоссы [1]:
  * CIoU loss [1]
* Кто больше? [5]
  * 0.05 mAP [1]
  * 0.1 mAP  [2]
  * 0.2 mAP [5]

**Максимальный балл:** 10 баллов. (+3 балла бонус).

In [15]:
import torch
import numpy as np
import pandas as pd
import albumentations as A

from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset
from albumentations.pytorch.transforms import ToTensorV2

from torchvision.models.detection.anchor_utils import AnchorGenerator

import io

### Загрузка данных

Мы продолжаем работу с датасетом из семинара - Halo infinite ([сслыка](https://universe.roboflow.com/graham-doerksen/halo-infinite-angel-aim)). Загрузка данных и создание датасета полностью скопированы из семинара.

Сначала загружаем данные

In [16]:
splits = {'train': 'data/train-00000-of-00001-0d6632d599c29801.parquet',
          'validation': 'data/validation-00000-of-00001-c6b77a557eeedd52.parquet',
          'test': 'data/test-00000-of-00001-866d29d8989ea915.parquet'}
df_train = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["train"])
df_test = pd.read_parquet("hf://datasets/Francesco/halo-infinite-angel-videogame/" + splits["test"])

Создаем датасет для предобработки данных

In [17]:
class HaloDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        df_objects = pd.json_normalize(dataframe['objects'])[["bbox", "category"]]
        df_images = pd.json_normalize(dataframe['image'])[["bytes"]]
        self.data = dataframe[["image_id"]].join(df_objects).join(df_images)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """Загружаем данные и разметку для объекта с индексом `idx`.

        labels: List[int] Набор классов для каждого ббокса,
        boxes: List[List[int]] Набор ббоксов в формате (x_min, y_min, w, h).
        """
        row = self.data.iloc[idx]
        image = Image.open(io.BytesIO(row["bytes"]))
        image = np.array(image)

        target = {}
        target["image_id"] = row["image_id"]

        labels = [row["category"]] if isinstance(row["category"], int) else row['category']
        # Вычитаем единицу чтобы классы начинались с нуля
        labels = [label - 1 for label in labels]
        boxes = row['bbox'].tolist()

        if self.transform is not None:
            transformed = self.transform(image=image, bboxes=boxes, labels=labels)
            image, boxes, labels = transformed["image"], transformed["bboxes"], transformed["labels"]
        else:
            image = transforms.ToTensor()(image)

        # Переводим боксы в формат xyxy
        boxes = np.array(boxes).reshape(-1, 4)
        if len(boxes) > 0:
            boxes[:, 2] = boxes[:, 0] + boxes[:, 2]
            boxes[:, 3] = boxes[:, 1] + boxes[:, 3]

        target['boxes'] = torch.tensor(boxes, dtype=torch.float32)
        target['labels'] = torch.tensor(labels, dtype=torch.int64)
        return image, target

def collate_fn(batch):
    batch = tuple(zip(*batch))
    images = torch.stack(batch[0])
    return images, batch[1]

Чтобы модель не переобучалась, можно добавить больше аугментаций, весь список можно посмотреть тут [[ссылка](https://explore.albumentations.ai/)].

Какие можно использовать аугментации?
* Добавить зум `RandomResizedCrop`,
* Сделать цветовые аугментации типа `RandomBrightnessContrast` и/или `HueSaturationValue`,
* Добавить шум `GaussNoise`,
* Вырезать случайные части изображения `CoarseDropout`,
* И любые другие!

Аугментации можно комбинировать посредствам `A.OneOf`, `A.SomeOf` или `A.RandomOrder`.

Хоть аугментации ограничиваются только вашей фантазией, перед обучением советуем посмотреть на результат преобразований и убедиться, что изображение ещё поддается детекции:)

In [18]:
mean = (0.485, 0.456, 0.406)
std = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    # Геометрия
    A.HorizontalFlip(p=0.5),
    A.Affine(
        scale=(0.85, 1.15),
        translate_percent=(-0.05, 0.05),
        rotate=(-5, 5),  # небольшой наклон камеры, не переворот
        p=0.7
    ),
    A.RandomSizedBBoxSafeCrop(
        height=640,
        width=640,
        erosion_rate=0.2,
        p=0.5
    ),
    A.Perspective(
        scale=(0.02, 0.05),
        keep_size=True,
        p=0.2
    ),

    # Освещение и контраст сцены (разные карты, время суток, эффекты выстрелов)
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.RandomGamma(gamma_limit=(80, 120), p=0.3),

    # ОСТОРОЖНО с цветом — минимальный сдвиг, чтобы не спутать команды
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=15, val_shift_limit=10, p=0.3),

    # Специфика динамичного шутера: смаз от быстрого движения камеры/спринта
    A.MotionBlur(blur_limit=5, p=0.2),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),

    # Шум (сжатие видео при записи геймплея, low bitrate стримов)
    A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
    A.ImageCompression(quality_range=(60, 95), p=0.3),

    # Окклюзия — эмуляция укрытий, других объектов на линии видимости
    A.CoarseDropout(
        num_holes_range=(1, 2),
        hole_height_range=(0.02, 0.05),
        hole_width_range=(0.02, 0.05),
        p=0.15
    ),

    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
], bbox_params=A.BboxParams(format='coco', label_fields=['labels'], min_visibility=0.3))

test_transform = A.Compose(
    [
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ]
)

Не забываем инициализировать наш датасет

In [19]:
train_dataset = HaloDataset(df_train, transform=train_transform)
test_dataset = HaloDataset(df_test, transform=test_transform)

## Переделываем модель [4 балла]

В семинаре мы реализовали самый базовый детектор, а сейчас настало время его улучшать.

### Backbone [1 балл]

Хорошей практикой считается размораживать несколько последних слоев в backbone, это позволяет немного улучить качество модели. Давайте улушчим класс Backbone из лекции, добавив ему возможность разморозки __k__ последних слоев или блоков (на ваш выбор).

In [20]:
import timm
import torch.nn as nn

In [21]:
class Backbone(nn.Module):
    def __init__(self, model_name="darknetaa53", unfreeze_last=3, out_indices=(-3, -2, -1)):
        super().__init__()
        self.model = timm.create_model(model_name=model_name, features_only=True, out_indices=out_indices)

        # замораживаем все
        for param in self.model.parameters():
            param.requires_grad = False

        # размораживаем последние слои
        if unfreeze_last > 0:
            params_list = list(self.model.parameters())
            for param in params_list[-unfreeze_last:]:
                param.requires_grad = True

    def forward(self, x):
        return self.model(x)

### NECK [2 балла]

Следующее улучшение коснется шеи. Предлагаем реализовать знакомую из лекции архитектуру FPN.

#### Feature Pyramid Network

<center><img src="https://user-images.githubusercontent.com/57972646/69858594-b14a6c00-12d5-11ea-8c3e-3c17063110d3.png"/></center>


* [Feature Pyramid Networks for Object Detection](https://arxiv.org/abs/1612.03144)

Она состоит из top-down пути, в котором происходит 2 вещи:
1. Увеличивается пространственная размерность фичей,
2. С помощью скипконнекшеннов, добавляются фичи из backbone модели.

Для увеличения пространственной размерности используется __nearest neighbor upsampling__, а фичи из шеи и бекбоуна суммируются.

__TIPS__:
* Можете использовать базовые классы из лекции,
* Воспользуйтесь AnchorGenerator-ом, чтобы создавать якоря сразу для нескольких выходов,
* Не забудьте использовать nn.ModuleList, если захотите сделать динамическое количество голов у модели,
* Также, можно добавить доп конволюцию (3х3 с паддингом) у каждого выхода шеи.

In [22]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class Neck(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        assert len(in_channels) == len(out_channels)
        self.length = len(in_channels)
        # lateral 1x1 convs — приводят каждый уровень backbone к единому out_channels
        self.laterals = nn.ModuleList([
            nn.Conv2d(in_channels[i], out_channels[i], kernel_size=1) for i in range(self.length)
        ])
        # 3x3 conv после сложения — сглаживает артефакты upsampling (стандартная практика FPN)
        self.smooths = nn.ModuleList([
            ConvBlock(out_channels[i], out_channels[i]) for i in range(self.length)
        ])
        self.upsample = nn.Upsample(scale_factor=2, mode='nearest')

    def forward(self, features):
        assert len(features) == self.length
        laterals = [self.laterals[i](features[i]) for i in range(self.length)]

        x = laterals[-1]
        result = [self.smooths[-1](x)]
        for i in range(self.length - 2, -1, -1):
            x = self.upsample(x) + laterals[i]   # сложение, не concat — каналы уже совпадают
            result.append(self.smooths[i](x))
        return result[::-1]  # разворачиваем, чтобы порядок совпадал с features (shallow -> deep)

In [23]:
class PAN(nn.Module):
    def __init__(self,  in_channels, out_channels):
        super().__init__()
        self.topdown = Neck(in_channels, out_channels)
        self.length = len(in_channels)
        self.smooths = nn.ModuleList([ConvBlock(out_channels[i], out_channels[i]) for i in range(self.length)])
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        y = self.topdown(x)
        x = y[0]
        out = [self.smooths[0](x)]
        
        for i in range(1, self.length):
            x = self.pool(out[-1])
            x = self.smooths[i](x + y[i])
            out.append(x)
        return out


### Head [1 балл]

В качестве шеи можно выбрать __один из двух__ вариантов:

#### 1. Decoupled Head

Реализовать Decoupled Head из [YOLOX](https://arxiv.org/abs/2107.08430).
<center><img src="https://i.ibb.co/BVtBR2R3/Decoupled-head.jpg"/></center>

**TIP**: Возьмите за основу голову из семинара, тк она сильно похожа на Decoupled Head.

Изменять количество параметров у шей на разных уровнях не обязательно.

#### 2. Confidence score free head

Нужно взять за основу голову из семинара и полностью убрать предсказание confidence score. Чтобы модель предсказывала только 2 группы: ббоксы и классы.

Есть следующие способы удаления confidence score:
* Добавление нового класса ФОН. Обычно его обозначают нулевым классом.
* Присваивание ббоксам БЕЗ объекта вектор из нулей в качестве таргета.

Выберете тот, который вам больше нравится и будте внимательны при расчете лосса!

**Важно!** Удаление confidence score повлияет на следующие методы из семинара:
* target_assign
* ComputeLoss
* _filter_predictions

In [24]:
# вариант YOLOX
class Head(nn.Module):
    def __init__(self, in_channels, num_anchors, num_classes):
        super().__init__()
        self.conv = ConvBlock(in_channels, 256, kernel_size=1, padding=0)
        self.cls_head = nn.Sequential(
            ConvBlock(256, 256),
            ConvBlock(256, 256),
            nn.Conv2d(256, num_classes * num_anchors, kernel_size=1, padding=0),
        )

        self.reg_part = nn.Sequential(
            ConvBlock(256, 256),
            ConvBlock(256, 256),
        )
        
        self.reg_head = nn.Conv2d(256, 4 * num_anchors, kernel_size=1, padding=0)
        self.iou_head = nn.Conv2d(256, num_anchors, kernel_size=1, padding=0)

    def forward(self, x):
        x = self.conv(x)
        cls = self.cls_head(x)
        reg_part = self.reg_part(x)
        reg = self.reg_head(reg_part)
        iou = self.iou_head(reg_part)

        return cls, iou, reg

Теперь можно снова реализовать класс детектора с учетом всех частей выше!

In [25]:
class Detector(nn.Module):
    def __init__(self, 
                 backbone_model_name="darknetaa53",
                 neck_class=Neck,
                 head_class=Head,
                 unfreeze_last_backbone=2,
                 out_indices_backbone=(-3, -2, -1),
                 neck_n_channels=256,
                 num_classes=4,
                 anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
                 anchor_ratios=((0.5, 1, 2),)*3,
                 input_size=(640, 640)
                ):
        super().__init__()
        self.num_outs = len(out_indices_backbone)
        self.num_classes = num_classes
        self.input_size = input_size
        self.backbone = Backbone(model_name=backbone_model_name, 
                                 unfreeze_last=unfreeze_last_backbone, 
                                 out_indices=out_indices_backbone)
        self.neck = neck_class(in_channels=self.backbone.model.feature_info.channels(), 
                         out_channels=[neck_n_channels]*self.num_outs)
        # Одна голова, общая для всех уровней (стандартная практика в YOLOX/RetinaNet —
        # веса шарятся между уровнями, это резко уменьшает число параметров головы)
        num_anchors = len(anchor_sizes[0]) * len(anchor_ratios[0])
        self.head = head_class(in_channels=neck_n_channels, num_anchors=num_anchors, num_classes=num_classes)

        # генерируем якоря для каждого уровня отдельно
        anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_ratios)
        reductions = self.backbone.model.feature_info.reduction()
        grid_sizes = []
        strides = []
        for reduction in reductions:
            grid_sizes.append([input_size[0] // reduction, input_size[1] // reduction])
            strides.append([reduction, reduction])

        anchors_per_level = anchor_generator.grid_anchors(grid_sizes, strides=strides)
        
        # Не стакаем в единый тензор — уровни имеют разное число якорей (разное H*W),
        # поэтому сохраняем как список буферов, а не как один тензор с общей размерностью
        for i, anchors in enumerate(anchors_per_level):
            anchor_centers = (anchors[:, :2] + anchors[:, 2:]) / 2
            anchor_size = (anchors[:, 2:] - anchors[:, :2])
            self.register_buffer(f"anchors_{i}", anchors)
            self.register_buffer(f"anchor_centers_{i}", anchor_centers)
            self.register_buffer(f"anchor_sizes_{i}", anchor_size)

    def forward(self, x):
        features = self.backbone(x)
        neck_features = self.neck(features)

        all_bbox_offsets, all_confidence_logits, all_cls_logits = [], [], []
        N = x.shape[0]
        
        for level, feat in enumerate(neck_features):
            cls_logits, iou_logits, bbox_preds = self.head(feat)

            cls_logits = cls_logits.permute(0, 2, 3, 1).contiguous().view(N, -1, self.num_classes)
            bbox_preds = bbox_preds.permute(0, 2, 3, 1).contiguous().view(N, -1, 4)
            iou_logits = iou_logits.permute(0, 2, 3, 1).contiguous().view(N, -1)

            all_cls_logits.append(cls_logits)
            all_bbox_offsets.append(bbox_preds)
            all_confidence_logits.append(iou_logits)

        # Склеиваем предсказания со всех уровней в единую последовательность якорей —
        # порядок должен совпадать с порядком anchors_per_level, зарегистрированных в __init__
        cls_logits = torch.cat(all_cls_logits, dim=1)
        bbox_offsets = torch.cat(all_bbox_offsets, dim=1)
        confidence_logits = torch.cat(all_confidence_logits, dim=1)
        
        if self.training:
            return bbox_offsets, confidence_logits, cls_logits

        bbox_preds = self.decode_boxes(bbox_offsets)
        iou_scores = torch.sigmoid(confidence_logits)
        cls_preds = torch.softmax(cls_logits, dim=-1)
        return bbox_preds, iou_scores, cls_preds

    def decode_boxes(self, bbox_preds: torch.Tensor) -> torch.Tensor:
        """Переводит смещения детектора (dx, dy, dw, dh) в координаты [x1, y1, x2, y2].
    
        bbox_preds: [N, Total_Anchors, 4]
        Возвращает:  [N, Total_Anchors, 4] в пикселях входного разрешения
        """
        anchor_centers = torch.cat([getattr(self, f"anchor_centers_{i}") for i in range(self.num_outs)], dim=0)
        anchor_sizes = torch.cat([getattr(self, f"anchor_sizes_{i}") for i in range(self.num_outs)], dim=0)
        
        dx = bbox_preds[..., 0]
        dy = bbox_preds[..., 1]
        dw = torch.clamp(bbox_preds[..., 2], max=4.0)  # защита от переполнения exp
        dh = torch.clamp(bbox_preds[..., 3], max=4.0)
    
        # 1. Центры и стороны предсказанных рамок
        pred_ctr_x = anchor_centers[:, 0] + torch.sigmoid(dx) * anchor_sizes[:, 0]
        pred_ctr_y = anchor_centers[:, 1] + torch.sigmoid(dy) * anchor_sizes[:, 1]
        pred_w = anchor_sizes[:, 0] * torch.exp(dw)
        pred_h = anchor_sizes[:, 1] * torch.exp(dh)
    
        # 2. Перевод в угол-угол [x1, y1, x2, y2]
        x1 = pred_ctr_x - 0.5 * pred_w
        y1 = pred_ctr_y - 0.5 * pred_h
        x2 = pred_ctr_x + 0.5 * pred_w
        y2 = pred_ctr_y + 0.5 * pred_h
    
        # 3. Обрезка по границам кадра
        x1 = x1.clamp(0, self.input_size[1])
        y1 = y1.clamp(0, self.input_size[0])
        x2 = x2.clamp(0, self.input_size[1])
        y2 = y2.clamp(0, self.input_size[0])
    
        return torch.stack([x1, y1, x2, y2], dim=-1)

    @property
    def anchors(self):
        """ Совместимость с Runner: единый тензор якорей со всех уровней подряд, формат xyxy. """
        return torch.cat([getattr(self, f"anchors_{i}") for i in range(self.num_outs)], dim=0)

## Label assignment [3 балла]
В этой секции предлагается заменить функцию `assign_target` на более современный алгоритм который называется Task alignment learning.

Он описан в статье [TOOD](https://arxiv.org/abs/2108.07755) в секции 3.2. Для удобства вот его основные шаги:

1. Посчитать значение метрики для каждого предсказанного ббокса:
    
$$t = s^\alpha * u^\beta$$
    
где,
* $s$ — classification score, или вероятность принадлежности предсказанного ббокса к классу реального ббокса (**GT**);
* $u$ — IoU между предсказанным и реальным ббоксами;
* $\alpha,\ \beta$ — нормализационные константы, обычно $\alpha = 6.0, \ \beta = 1.0$.
    
2. Отфильтровать предсказания на основе **GT**.

    Для якорных детекторов, обычно, выбираются только те предсказания, центры якорей которых находятся внутри GT.
4. Для каждого **GT** выбрать несколько (обычно 5 или 13) самых подходящих предсказаний.
5. Если предсказание рассматривается в качестве подходящего для нескольких **GT** — выбрать **GT** с наибольшим пересечением по IoU.


**BAЖНО**: если будете использовать Runner из лекции, не забудьте поменять параметры  в `self.assign_target_method` в методе `_run_train_epoch`.

In [26]:
from torchvision.ops import box_iou

def TAL_assigner(anchors, gt_boxes, gt_labels, cls_logits, num_classes, alpha=6.0, beta=1.0, k_best=13):
    """ TOOD-style Task Alignment Learning assigner. Считается для одной картинки.

    anchors, gt_boxes — формат xyxy.
    cls_logits — сырые логиты классификации для каждого якоря, форма (num_anchors, num_classes).

    Returns
    -------
    target_offsets: (num_anchors, 4) — GT-боксы в xyxy для позитивных якорей, 0 для остальных
    target_objectness: (num_anchors,) — 1 позитивный, 0 отрицательный
    target_cls: (num_anchors, num_classes) — one-hot классы для позитивных якорей
    """
    num_anchors = anchors.shape[0]
    target_objectness = torch.zeros(num_anchors, device=anchors.device)
    target_offsets = torch.zeros((num_anchors, 4), device=anchors.device)
    target_cls = torch.zeros((num_anchors, num_classes), device=anchors.device)

    if gt_boxes.numel() == 0:
        return target_offsets, target_objectness, target_cls

    # 1. Метрика выравнивания t = s^alpha * u^beta
    u = box_iou(anchors, gt_boxes)                         # [num_anchors, num_gt]
    s = cls_logits.softmax(dim=-1)[:, gt_labels]           # [num_anchors, num_gt]
    t = s**alpha * u**beta

    anchor_centers = (anchors[:, :2] + anchors[:, 2:]) / 2

    all_anchor_idxs, all_gt_idxs = [], []
    for i in range(gt_boxes.shape[0]):
        # 2. Фильтруем якоря, чьи центры внутри GT
        gt_mask = (gt_boxes[i, 0] <= anchor_centers[:, 0]) & (anchor_centers[:, 0] <= gt_boxes[i, 2]) & \
                  (gt_boxes[i, 1] <= anchor_centers[:, 1]) & (anchor_centers[:, 1] <= gt_boxes[i, 3])
        valid_idxs = gt_mask.nonzero(as_tuple=True)[0]

        # Fallback как в старой функции — если ни один якорь не попал внутрь GT
        if valid_idxs.numel() == 0:
            valid_idxs = u[:, i].argmax().unsqueeze(0)

        # 3. Топ-k_best по t среди отфильтрованных
        anchors_valid_t = t[valid_idxs, i]
        _, order = torch.sort(anchors_valid_t, descending=True)
        top_k_idxs = valid_idxs[order[:k_best]]

        all_anchor_idxs.append(top_k_idxs)
        all_gt_idxs.append(torch.full_like(top_k_idxs, i))

    all_anchor_idxs = torch.cat(all_anchor_idxs, dim=0)
    all_gt_idxs = torch.cat(all_gt_idxs, dim=0)

    # 4. Если якорь выбран для нескольких GT — оставляем GT с максимальным IoU
    ious_selected = u[all_anchor_idxs, all_gt_idxs]
    order = torch.argsort(ious_selected, descending=True)
    sorted_anchor_idxs = all_anchor_idxs[order]
    sorted_gt_idxs = all_gt_idxs[order]

    _, first_idx = np.unique(sorted_anchor_idxs.cpu().numpy(), return_index=True)
    first_idx = torch.as_tensor(sorted(first_idx), device=anchors.device)

    pos_anchor_idxs = sorted_anchor_idxs[first_idx]
    pos_gt_idxs = sorted_gt_idxs[first_idx]

    # 5. Заполняем таргеты в том же формате, что и старая assign_target
    target_objectness[pos_anchor_idxs] = 1
    target_offsets[pos_anchor_idxs] = gt_boxes[pos_gt_idxs]          # xyxy-координаты GT, не offset!
    target_cls[pos_anchor_idxs, gt_labels[pos_gt_idxs]] = 1

    return target_offsets, target_objectness, target_cls

### DIoU [1]

Вместо SmoothL1, который используется в семинаре, реализуем лосс, основанный на пересечении ббоксов. В качестве тренировки давайте напишем Distance Intersection over Union (DIoU).

<center><img src=https://wikidocs.net/images/page/163613/Free_Fig_5.png></center>

Для его реализации разобъем задачу на части:

**1. Реализуем IoU:**

Пусть даны координаты для предсказанного ($B^p$) и истинного ($B^g$) ббоксов в формате XYXY или VOC PASCAL (левый верхний и правый нижний углы):

$B^p=(x^p_1, y^p_1, x^p_2, y^p_2)$, $B^g=(x^g_1, y^g_1, x^g_2, y^g_2)$, тогда алгоритм расчета будет следующий:

    1. Найдем площади обоих ббоксов:
$$ A^p = (x^p_2 - x^p_1) * (y^p_2 - y^p_1) $$
$$ A^g = (x^g_2 - x^g_1) * (y^g_2 - y^g_1) $$

    2. Посчитаем пересечение между ббоксами:

Тут мы предлагаем вам подумать как в общем виде можно расчитать размеры ббокса, который будет являться пересечением $B^p$ и $B^g$, а затем посчитать его площадь:

$$x^I_1 = \qquad \qquad y^I_1 = $$
$$x^I_2 = \qquad \qquad y^I_2 = $$

В общем виде, площать будет записываться следующим образом:

Если $x^I_2 > x^I_1$ & $y^I_2 > y^I_1$, тогда:

$$I = (x^I_2 - x^I_1) * (y^I_2 - y^I_1)$$

Иначе, $I = 0$.

    3. Считаем объединение ббоксов.

Мы можем посчитать эту площадь как сумму площадей двух ббоксов минус площадь пересечения (тк мы считаем её два раз в сумме площадей):

$$U = A^p + A^g - I$$

    4. Вычисляем IoU.

$$IoU = \frac{I}{U}$$

**2. Посчитаем диагональ выпуклой оболочки:**

Для расчета диагонали, сначала выпишите координаты верхнего левого и правого нижнего углов. Подумайте, чему будут равны эти координаты в общем случае?

$$x^c_1 = \qquad \qquad y^c_1 = $$
$$x^c_2 = \qquad \qquad y^c_2 = $$

Подсказка: Нарисуйте несколько вариантов пересечений предсказания и GT на бумажке, и выпишите координаты для выпуклой оболочки.

Тогда квадрат диагонали можно посчитать по формуле:

$$c^2 = (x^c_2 - x^c_1)^2 + (y^c_2 - y^c_1)^2$$

**3. Рассчитаем расстояние между цетрами ббоксов:**

Сначала находим координаты центров каждого из ббоксов (если ббоксы в формате YOLO, то и считать ничего не нужно), затем считаем Евклидово расстояние между центрами.

$d = $

Собираем все части вместе и считаем лосс по формуле:

$$ DIoU = 1 - IoU + \frac{d^2}{c^2}$$

Помните, что пар ббоксов может быть много! Возвращайте усредненное значение лосса.

In [27]:
from torchvision.ops import distance_box_iou_loss

In [28]:
def gen_bbox(num_boxes=10):
    min_corner = torch.randint(0, 100, (num_boxes, 2))
    max_corner = torch.randint(50, 150, (num_boxes, 2))

    for i in range(2):
        wrong_order = min_corner[:, i] > max_corner[:, i]
        if wrong_order.any():
            min_corner[wrong_order, i], max_corner[wrong_order, i] = max_corner[wrong_order, i], min_corner[wrong_order, i]
    return torch.cat((min_corner, max_corner), dim=1)

In [29]:
pred_boxes = gen_bbox(num_boxes=100)
true_boxes = gen_bbox(num_boxes=100)

In [30]:
print(f" DIoU: {distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean").item()}")

 DIoU: 1.0202234983444214


In [31]:
def diou_loss(pred_boxes, gt_boxes):
    Ap = (pred_boxes[:, 2] - pred_boxes[:, 0]) * (pred_boxes[:, 3] - pred_boxes[:, 1])
    Ag = (gt_boxes[:, 2] - gt_boxes[:, 0]) * (gt_boxes[:, 3] - gt_boxes[:, 1])

    xDiff = torch.min(torch.stack([pred_boxes[:, 2], gt_boxes[:, 2]], dim=1), dim=1)[0] - torch.max(torch.stack([pred_boxes[:, 0], gt_boxes[:, 0]], dim=1), dim=1)[0]
    yDiff = torch.min(torch.stack([pred_boxes[:, 3], gt_boxes[:, 3]], dim=1), dim=1)[0] - torch.max(torch.stack([pred_boxes[:, 1], gt_boxes[:, 1]], dim=1), dim=1)[0]
    I = xDiff.clamp(min=0) * yDiff.clamp(min=0)

    U = Ap + Ag - I
    iou = I / (U + 1e-7)

    xcDiff = torch.max(torch.stack([pred_boxes[:, 2], gt_boxes[:, 2]], dim=1), dim=1)[0] - torch.min(torch.stack([pred_boxes[:, 0], gt_boxes[:, 0]], dim=1), dim=1)[0]
    ycDiff = torch.max(torch.stack([pred_boxes[:, 3], gt_boxes[:, 3]], dim=1), dim=1)[0] - torch.min(torch.stack([pred_boxes[:, 1], gt_boxes[:, 1]], dim=1), dim=1)[0]
    c2 = xcDiff ** 2 + ycDiff ** 2

    pred_boxes_centers = (pred_boxes[:, 2:] + pred_boxes[:, :2]) / 2
    gt_boxes_centers = (gt_boxes[:, 2:] + gt_boxes[:, :2]) / 2
    d = nn.functional.pairwise_distance(pred_boxes_centers, gt_boxes_centers)

    diou = 1 - iou + d**2 / c2
    return diou.mean()

In [32]:
import numpy as np
pred_boxes = gen_bbox(num_boxes=1000)
true_boxes = gen_bbox(num_boxes=1000)

# проверим что написанный лосс выдает те же результаты что и лосс из торча.
assert np.isclose(diou_loss(pred_boxes, true_boxes), distance_box_iou_loss(pred_boxes, true_boxes, reduction="mean"))

## Кто больше? [5 баллов]

Наконец то мы дошли до самый интересной части. Тут мы раздаем очки за mAP'ы!

Все что вы написали выше вам поможет улучшить качество итогового детектора, настало время узнать насколько сильно :)

За достижения порога по mAP на тестовом наборе вы получаете баллы:
* 0.05 mAP [1]
* 0.1 mAP [2]
* 0.2 mAP [5]


**TIPS**:
1. На семинаре мы специально не унифицировали формат ббоксов между методами, чтобы обратить ваше внимание что за этим нужно следить. Чтобы было проще, сразу унифицируете формат по всему ноутбуку. Советуем использовать формат xyxy, тк IoU и NMS из torch используют именно этот формат. (Не забудьте поменять формат у таргета в `HaloDataset`).

2. Попробуйте перейти к IoU-based лоссу при обучении. То есть обучать не смещения, а сразу предсказывать ббокс.

3. Поэксперементируйте с подходами target assignment'а в процессе обучения. Например, можно на первых итерациях использовать обычный метод, а затем подключить TAL.

4. Добавьте аугментаций!

Можно взять [albumentations](https://albumentations.ai/docs/getting_started/bounding_boxes_augmentation/), библиотеку, которую мы использовали всеминаре. Или базовые аугментации из торча [тык](https://pytorch.org/vision/main/transforms.html). Если будете использовать торч, не забудте про ббоксы, transforms из коробки не будет их агументировать.

5. Можете реализовать другую шею, которую мы обсуждали на лекции [Path Aggregation Network](https://arxiv.org/abs/1803.01534) она точно улучшит ваше итоговое качество.

6. Попробуйте добавлять различные блоки из YOLO архитектур в шею вместо единичных конволюционных слоев. (Например, замените конволюции 3х3 на CSP блоки).

7. Попробуйте заменить NMS на другой метод (WeightedNMS, SoftNMS, etc.). Немного ссылок:
    * Статья про SoftNMS [тык](https://arxiv.org/pdf/1704.04503)
    * Статья про WeightedNMS [тык](https://openaccess.thecvf.com/content_ICCV_2017_workshops/papers/w14/Zhou_CAD_Scale_Invariant_ICCV_2017_paper.pdf)
    * Есть их реализация, правда на нумбе [git](https://github.com/ZFTurbo/Weighted-Boxes-Fusion?tab=readme-ov-file)

8. Не бойтесь эксперементировать и удачи!

Также, напишите развернутые ответы на следующие вопросы:

**Questions:**
1. Какой метод label assignment'a помогает лучше обучаться модели? Почему?
2. Какое из сделаных вами улучшений внесло наибольший вклад в качество модели? Как вы думаете, почему это произошло?
3. Какое из сделанных вами улучшений вообще не изменило метрику? Как вы думаете, почему это произошло?

In [33]:
class ComputeLoss:
    """ Базовый расчет лосса.

    Параметры
    ---------
    bbox_loss : Локализационная часть лосса
    obj_loss : Лосс для Confidence score
    cls_loss : Классификационная часть лосса
    weight_bbox, weight_obj, weight_cls : Константы для баллансировки частей лосса
    """
    def __init__(self,
            bbox_loss=None, obj_loss=None, cls_loss=None,
            weight_bbox=5, weight_obj=1, weight_cls=1
        ):
        self.bbox_loss = nn.SmoothL1Loss() if bbox_loss is None else bbox_loss
        self.obj_loss = nn.BCEWithLogitsLoss() if obj_loss is None else obj_loss
        self.cls_loss = nn.BCEWithLogitsLoss() if cls_loss is None else cls_loss
        self.weight_bbox = weight_bbox
        self.weight_obj = weight_obj
        self.weight_cls = weight_cls

    def __call__(self, predicts, targets):
        """ Расчет лосса для пары (предсказание, таргет)

        Параметры
        ---------
        predicts : Предсказания модели для одной картинки: Смещения, objectness score и логиты для классов
        targets : Gt значения для расчета лосса, а именно: GT смещения, GT objectness score и GT ohe классы
        """
        pred_offsets, pred_obj_logits, pred_cls_logits = predicts
        target_boxes, target_obj, target_cls = targets
        # Confidence score считается только для предсказаний соотв отрицательным и положительным якорям
        valid_mask = target_obj != -1
        loss_obj = self.obj_loss(pred_obj_logits[valid_mask], target_obj[valid_mask])

        # Локализационная и классификационные части считаются для предсказаинй соотв положительным якорям
        pos_mask = target_obj == 1
        if pos_mask.sum() > 0:
            loss_cls = self.cls_loss(pred_cls_logits[pos_mask], target_cls[pos_mask])
            loss_bbox = self.bbox_loss(pred_offsets[pos_mask], target_boxes[pos_mask])
        else:
            loss_cls = torch.tensor(0.0, device=pred_offsets.device)
            loss_bbox = torch.tensor(0.0, device=pred_offsets.device)
        return self.weight_bbox * loss_bbox + self.weight_obj * loss_obj + self.weight_cls * loss_cls

In [6]:
!pip install mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 691.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 37.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 50.1 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [7]:
import os
import dagshub
import mlflow
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["DAGSHUB_CLIENT_TOKEN"] = user_secrets.get_secret("DAGSHUB_TOKEN")

dagshub.init(repo_owner="egrsid", repo_name="detection_from_scratch", mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=e88546e3-40c6-4b4d-b3c1-ae5a635cffdc&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=f5ec9b7c85276380ee96659eebbad356ba0808dff8308f349e990d9de24da0c1




Accessing as egrsid

Initialized MLflow to track repo "egrsid/detection_from_scratch"

Repository egrsid/detection_from_scratch initialized!

In [34]:
import os
import torch
import torch.nn as nn
import numpy as np
import mlflow
from functools import partial
from tqdm.auto import tqdm
from torchvision.ops import nms
from torchmetrics.detection import MeanAveragePrecision
import matplotlib.pyplot as plt

### Weighted NMS

In [35]:
def weighted_nms(boxes, scores, iou_threshold=0.5):
    if boxes.numel() == 0:
        return boxes, scores

    merged_boxes = []
    merged_scores = []
    remaining_idxs = torch.arange(boxes.shape[0], device=boxes.device)

    while remaining_idxs.numel() > 0:
        cur_boxes = boxes[remaining_idxs]
        cur_scores = scores[remaining_idxs]

        top_idx = torch.argmax(cur_scores)
        top_box = cur_boxes[top_idx].unsqueeze(0)

        ious = box_iou(top_box, cur_boxes).squeeze(0)
        ious = torch.nan_to_num(ious, nan=0.0)  # <-- защита от NaN

        merge_mask = ious >= iou_threshold
        merge_mask[top_idx] = True  # <-- лидер ВСЕГДА должен попадать в свою группу

        group_boxes = cur_boxes[merge_mask]
        group_scores = cur_scores[merge_mask]

        weights = group_scores.unsqueeze(1)
        weighted_box = (group_boxes * weights).sum(dim=0) / weights.sum().clamp(min=1e-8)

        merged_boxes.append(weighted_box)
        merged_scores.append(group_scores.max())

        keep_local_mask = ~merge_mask
        remaining_idxs = remaining_idxs[keep_local_mask]

    return torch.stack(merged_boxes, dim=0), torch.stack(merged_scores, dim=0)

In [36]:
def _filter_predictions(predictions, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8, return_type="list"):
    bboxes, confidences, cls_probs = predictions
    all_final_scores = confidences[:, :, np.newaxis] * cls_probs

    num_classes = cls_probs.shape[-1]
    final_predictions = []
    for boxes, final_scores in zip(bboxes, all_final_scores):
        preds = {"boxes": [], "labels": [], "scores": []}
        for cls in range(num_classes):
            cls_scores = final_scores[:, cls]
            keep_ixs = cls_scores > score_threshold
            if keep_ixs.sum() == 0:
                continue
            cls_boxes = boxes[keep_ixs]
            cls_scores = cls_scores[keep_ixs]

            if len(cls_boxes) > max_boxes_per_cls * 4:  # берём с запасом, weighted NMS сам сократит
                pos = torch.argsort(cls_scores, descending=True)
                cls_boxes = cls_boxes[pos[:max_boxes_per_cls * 4]]
                cls_scores = cls_scores[pos[:max_boxes_per_cls * 4]]

            # Заменили nms(...) на weighted_nms(...)
            merged_boxes, merged_scores = weighted_nms(cls_boxes, cls_scores, nms_threshold)

            # Ограничиваем итоговое число боксов после объединения
            if len(merged_boxes) > max_boxes_per_cls:
                top_idxs = torch.argsort(merged_scores, descending=True)[:max_boxes_per_cls]
                merged_boxes = merged_boxes[top_idxs]
                merged_scores = merged_scores[top_idxs]

            for box, score in zip(merged_boxes, merged_scores):
                preds["boxes"].append(box.cpu().tolist())
                preds["labels"].append(cls)
                preds["scores"].append(score.item())

        if return_type == "torch":
            for key, item in preds.items():
                preds[key] = torch.tensor(item)
        elif return_type != "list":
            raise ValueError(f"Received unexpected `return_type`. Could be either `torch` or `list`, not {return_type}")
        final_predictions.append(preds)
    return final_predictions

### Default NMS

In [ ]:
def _filter_predictions(predictions, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8, return_type="list"):
    """ Ббоксы в `predictions` в формате (x_min, y_min, x_max, y_max). """
    bboxes, confidences, cls_probs = predictions
    all_final_scores = confidences[:, :, np.newaxis] * cls_probs

    num_classes = cls_probs.shape[-1]
    final_predictions = []
    for boxes, final_scores in zip(bboxes, all_final_scores):
        preds = {"boxes": [], "labels": [], "scores": []}
        for cls in range(num_classes):
            cls_scores = final_scores[:, cls]
            keep_ixs = cls_scores > score_threshold
            if keep_ixs.sum() == 0:
                continue
            cls_boxes = boxes[keep_ixs]
            cls_scores = cls_scores[keep_ixs]

            if len(cls_boxes) > max_boxes_per_cls:
                pos = torch.argsort(cls_scores, descending=True)
                cls_boxes = cls_boxes[pos[:max_boxes_per_cls]]
                cls_scores = cls_scores[pos[:max_boxes_per_cls]]

            pred_ixs = nms(cls_boxes, cls_scores, nms_threshold)
            for ix in pred_ixs:
                preds["boxes"].append(cls_boxes[ix].cpu().tolist())
                preds["labels"].append(cls)
                preds["scores"].append(cls_scores[ix].item())
        if return_type == "torch":
            for key, item in preds.items():
                preds[key] = torch.tensor(item)
        elif return_type != "list":
            raise ValueError(f"Received unexpected `return_type`. Could be either `torch` or `list`, not {return_type}")
        final_predictions.append(preds)
    return final_predictions

In [37]:
class Runner:
    """ Класс для обучения и валидации детектора с логированием в MLflow,
    early stopping'ом и устойчивым сохранением чекпоинтов.

    Параметры
    ---------
    name_to_save : str
        Базовое имя для файлов чекпоинтов (например, "halo_detector").
    checkpoint_dir : str
        Папка, куда сохраняются чекпоинты.
    patience : int
        Число эпох без улучшения val-метрики, после которого срабатывает early stopping.
    min_delta : float
        Минимальное улучшение метрики, чтобы считать эпоху "лучшей".
    """
    def __init__(self, model, compute_loss, optimizer, train_dataloader, assign_target_method,
                 name_to_save, checkpoint_dir="/kaggle/working/checkpoints",
                 device=None, scheduler=None, assign_target_kwargs=None,
                 val_dataloader=None, val_every=1, score_threshold=0.1, nms_threshold=0.5,
                 max_boxes_per_cls=8, patience=3, min_delta=0.002):
        # Сохраняем аргументы, с которыми был создан Runner (до создания self.* атрибутов)
        init_args = {k: v for k, v in locals().items() if k != "self"}
        self.init_kwargs = {
            k: (v if isinstance(v, (int, float, str, bool, type(None))) else repr(v))
            for k, v in init_args.items()
        }
        
        self.model = model
        self.compute_loss = compute_loss
        self.optimizer = optimizer
        self.train_dataloader = train_dataloader
        assign_target_kwargs = {} if assign_target_kwargs is None else assign_target_kwargs
        self.assign_target_method = partial(assign_target_method, **assign_target_kwargs)
        self.device = "cpu" if device is None else device
        self.scheduler = scheduler

        self.val_dataloader = val_dataloader
        self.val_every = val_every
        self.score_threshold = score_threshold
        self.nms_threshold = nms_threshold
        self.max_boxes_per_cls = max_boxes_per_cls

        self.name_to_save = name_to_save
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)

        self.patience = patience
        self.min_delta = min_delta
        self.epochs_without_improvement = 0
        self.best_val_metric = -float("inf")
        self.best_checkpoint_path = None

        self.batch_loss = []
        self.epoch_loss = []
        self.val_metric = []

    @property
    def raw_model(self):
        """ Достаёт исходную модель из-под nn.DataParallel, если она обёрнута. """
        return self.model.module if isinstance(self.model, nn.DataParallel) else self.model

    def _checkpoint_path(self, tag):
        return os.path.join(self.checkpoint_dir, f"{self.name_to_save}_{tag}.pt")

    def _save_checkpoint(self, tag="last"):
        """ Сохраняем веса модели (без обёртки DataParallel) и оптимизатор. """
        path = self._checkpoint_path(tag)
        torch.save({
            "model_state_dict": self.raw_model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "epoch_loss": self.epoch_loss,
            "val_metric": self.val_metric,
        }, path)
        return path

    def _run_train_epoch(self, dataloader, verbose=True):
        self.model.train()
        batch_loss = []
        epoch_pos_ious = []  # <-- копим IoU по всем батчам эпохи
        for images, targets in (pbar := tqdm(dataloader, desc="Process train epoch", leave=False)):
            images = images.to(self.device)
            outputs = self.model(images)

            anchors = self.raw_model.anchors.view(-1, 4)
            accum_loss = 0.0
            batch_pos_ious = []  # <-- IoU внутри текущего батча
            for ix in range(images.shape[0]):
                gt_boxes = targets[ix]['boxes'].to(self.device)
                gt_labels = targets[ix]['labels'].to(self.device)

                outputs_ixs = [out[ix] for out in outputs]
                pred_offsets, pred_obj_logits, pred_cls_logits = outputs_ixs

                assigned_targets = self.assign_target_method(
                    anchors, gt_boxes, gt_labels, pred_cls_logits, num_classes=self.raw_model.num_classes
                )
                target_offsets, target_obj, target_cls = assigned_targets

                decoded_boxes = self.raw_model.decode_boxes(pred_offsets.unsqueeze(0)).squeeze(0)
                loss = self.compute_loss((decoded_boxes, pred_obj_logits, pred_cls_logits), assigned_targets)
                accum_loss += loss

                # Замер IoU между декодированными позитивными предсказаниями и назначенными GT
                pos_mask = target_obj == 1
                if pos_mask.sum() > 0:
                    with torch.no_grad():
                        ious = box_iou(decoded_boxes[pos_mask], target_offsets[pos_mask])
                        pos_ious = ious.diagonal()  # каждый якорь со СВОИМ GT, а не со всеми
                        batch_pos_ious.append(pos_ious.mean().item())

            accum_loss = accum_loss / images.shape[0]
            batch_loss.append(accum_loss.cpu().detach().item())

            self.optimizer.zero_grad()
            accum_loss.backward()
            self.optimizer.step()

            # выводим iou
            if batch_pos_ious:
                mean_batch_iou = np.mean(batch_pos_ious)
                epoch_pos_ious.append(mean_batch_iou)
                mlflow.log_metric("train_pos_anchor_iou", round(mean_batch_iou, 4),
                                   step=len(self.batch_loss) + len(batch_loss))

            mlflow.log_metric("train_batch_loss", batch_loss[-1], step=len(self.batch_loss) + len(batch_loss))

            # тут тоже выводим iou
            if verbose:
                iou_str = f", pos_iou: {epoch_pos_ious[-1]:.4f}" if epoch_pos_ious else ""
                pbar.set_description(f"Current batch loss: {batch_loss[-1]:.4f}{iou_str}")
        return batch_loss

    def train(self, num_epochs=10, verbose=True):
        with mlflow.start_run(run_name=self.name_to_save):
            mlflow.log_params(self.init_kwargs)
            mlflow.log_param("num_epochs", num_epochs)
            mlflow.log_param("optimizer_name", type(self.optimizer).__name__)
            mlflow.log_param("initial_lr", self.optimizer.param_groups[0]["lr"])
            mlflow.log_param("loss_function", type(self.compute_loss).__name__)
            mlflow.log_param("patience", self.patience)

            try:
                for epoch in (epoch_pbar := tqdm(range(1, num_epochs + 1), desc="Train epoch", total=num_epochs)):
                    loss = self._run_train_epoch(self.train_dataloader, verbose=verbose)
                    self.batch_loss.extend(loss)
                    self.epoch_loss.append(np.mean(self.batch_loss[-len(self.train_dataloader):]))
                    mlflow.log_metric("train_epoch_loss", self.epoch_loss[-1], step=epoch)

                    if self.val_dataloader is not None and epoch % self.val_every == 0:
                        val_metric = self.validate()
                        self.val_metric.append(val_metric)
                        val_desc = f" Val {val_metric:.4}"
                        mlflow.log_metric("val_map", val_metric, step=epoch)

                        # Проверка на лучшую модель (аналог train_model, но по val_map)
                        if val_metric >= self.best_val_metric + self.min_delta:
                            self.best_val_metric = val_metric
                            self.epochs_without_improvement = 0
                            self.best_checkpoint_path = self._save_checkpoint(tag="best")
                        else:
                            self.epochs_without_improvement += 1

                    if verbose:
                        epoch_pbar.set_postfix({
                                "train_loss": f"{self.epoch_loss[-1]:.4f}",
                                "val_map": f"{self.val_metric[-1]:.4f}" if self.val_metric else "—",
                            })

                    if self.scheduler is not None:
                        self.scheduler.step()
                        mlflow.log_metric("lr", self.optimizer.param_groups[0]["lr"], step=epoch)

                    # Подстраховка на случай обрыва между эпохами (не только KeyboardInterrupt)
                    self._save_checkpoint(tag="last")

                    if self.val_dataloader is not None and self.epochs_without_improvement >= self.patience:
                        print(f"\n[Early Stopping] Нет улучшений {self.patience} эпох подряд, остановка на эпохе {epoch}.")
                        break

            except KeyboardInterrupt:
                print("\nОбучение прервано вручную — сохраняю текущие веса модели.")
            except Exception as e:
                print(f"\n[Ошибка во время обучения]: {e}")
                raise e
            finally:
                interrupted_path = self._save_checkpoint(tag="interrupted")
                mlflow.log_artifact(interrupted_path)
                if self.best_checkpoint_path and os.path.exists(self.best_checkpoint_path):
                    mlflow.log_artifact(self.best_checkpoint_path)
                    mlflow.set_tag("best_val_map", f"{self.best_val_metric:.4f}")
                print(f"Чекпоинты сохранены в {self.checkpoint_dir}")

        # Загружаем лучшие веса обратно в модель (только если не было настоящей ошибки)
        if self.best_checkpoint_path and os.path.exists(self.best_checkpoint_path):
            print(f"Загрузка лучших весов ({self.best_checkpoint_path}) в модель.")
            checkpoint = torch.load(self.best_checkpoint_path, map_location=self.device, weights_only=False)
            self.raw_model.load_state_dict(checkpoint["model_state_dict"])

    @torch.no_grad()
    def validate(self, dataloader=None):
        self.model.eval()
        dataloader = self.val_dataloader if dataloader is None else dataloader
        metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
        for images, targets in tqdm(dataloader, desc="Running validation", leave=False):
            images = images.to(self.device)
            outputs = self.model(images)
            predicts = _filter_predictions(outputs, self.score_threshold, self.nms_threshold,
                                            max_boxes_per_cls=self.max_boxes_per_cls, return_type="torch")
            metric.update(predicts, targets)
        return metric.compute()["map"].item()

    def plot_loss(self, row_figsize=3):
        nrows = 2 if self.val_metric else 1
        _, ax = plt.subplots(nrows, 1, figsize=(12, row_figsize * nrows), tight_layout=True)
        ax = np.array([ax]) if not isinstance(ax, np.ndarray) else ax

        ax[0].plot(self.batch_loss, label="Train batch Loss", color="tab:blue")
        epoch_x = np.arange(1, len(self.epoch_loss) + 1) * len(self.train_dataloader)
        ax[0].plot(epoch_x, self.epoch_loss, color="tab:orange", label="Train epoch Loss")
        ax[0].grid()
        ax[0].set_title("Train Loss")
        ax[0].set_xlabel("Number of Iterations")
        ax[0].set_ylabel("Loss")

        if self.val_metric:
            val_x = np.arange(1, len(self.val_metric) + 1) * self.val_every * len(self.train_dataloader)
            ax[1].plot(val_x, np.array(self.val_metric) * 100, color="tab:green", label="Validation mAP")
            ax[1].grid()
            ax[1].set_title("Validation mAP")
            ax[1].set_xlabel("Number of Iterations")
            ax[1].set_ylabel("mAP (%)")

        plt.legend()
        plt.show()

In [38]:
from torch.utils.data import DataLoader
import torch.optim as optim
from functools import partial
from tqdm.auto import tqdm

In [39]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, 
                              collate_fn=collate_fn, num_workers=2, pin_memory=True,
                             persistent_workers=True)
test_dataloader = DataLoader(test_dataset, batch_size=18, shuffle=False, 
                             collate_fn=collate_fn, num_workers=2, pin_memory=True,
                             persistent_workers=True)

In [40]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean()

### Darknetaa53 ul15

In [ ]:
m = timm.create_model('darknetaa53', features_only=True)
m.requires_grad_(False)
params = list(m.parameters())
for i, param in enumerate(params[::-1]):
    if i == 15: break
    param.requires_grad = True

torchinfo.summary(m, (1, 3, 640, 640))

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    unfreeze_last_backbone=15,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

In [ ]:
import torchinfo
torchinfo.summary(model.module, (1, 3, 640, 640))

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 1e-3},
    {'params': raw.neck.parameters(), 'lr': 1e-3},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

smooth_l1_loss = diou_loss
bce_loss = nn.BCEWithLogitsLoss()
ce_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(smooth_l1_loss, bce_loss, ce_loss, weight_bbox=5)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="darknetaa53_fpn_ul15_ep50", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 6, 'beta': 1, 'k_best': 13},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=8, min_delta=0.0004)

num_epochs = 50

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### Darknetaa53 ul15 больше эпох + трекинг iou между полижительными и gt боксами

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    unfreeze_last_backbone=15,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 1e-3},
    {'params': raw.neck.parameters(), 'lr': 1e-3},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

smooth_l1_loss = diou_loss
bce_loss = nn.BCEWithLogitsLoss()
ce_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(smooth_l1_loss, bce_loss, ce_loss, weight_bbox=5)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="darknetaa53_fpn_ul15_ep50_no_early_stopping", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 6, 'beta': 1, 'k_best': 20},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=50, min_delta=0.0003)

num_epochs = 50

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

In [ ]:
model.eval()
with torch.no_grad():
    bboxes, confidences, cls_probs = model(test_dataset[0][0].unsqueeze(0).to(device))
print(confidences.max().item(), confidences.mean().item())

### Darknetaa53 + FocalLoss + no early stopping

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    unfreeze_last_backbone=10,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 1e-3},
    {'params': raw.neck.parameters(), 'lr': 1e-3},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

reg_loss = diou_loss
obj_loss = FocalLoss()
cls_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(reg_loss, obj_loss, cls_loss, weight_bbox=5)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="darknetaa53_fpn_ul10_ep50_obj_focal", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 6, 'beta': 1, 'k_best': 20},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=50, min_delta=0.0003, score_threshold=0.05)

num_epochs = 50

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### Darknetaa53 + FocalLoss + no early stopping + ul5 + nums_threshold=0.1 + alpha=1, beta=6 ДООБУЧЕНИЕ

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    unfreeze_last_backbone=5,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)

RUN_ID = "829b5fdf21ea444986d983e6987ab3f7"
ARTIFACT_NAME = "darknetaa53_fpn_ul10_ep50_obj_focal_best.pt"
DOWNLOAD_DIR = "/kaggle/working/downloaded_models"

client = mlflow.tracking.MlflowClient()
local_weight_path = client.download_artifacts(
    run_id=RUN_ID,
    path=ARTIFACT_NAME,
    dst_path=DOWNLOAD_DIR
)

In [ ]:
state_dict = torch.load(local_weight_path, map_location=device, weights_only=False)
model.load_state_dict(state_dict["model_state_dict"])
model = model.to(device)

In [ ]:
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 1e-3},
    {'params': raw.neck.parameters(), 'lr': 1e-3},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=5e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-5)

reg_loss = diou_loss
obj_loss = FocalLoss()
cls_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(reg_loss, obj_loss, cls_loss, weight_bbox=7)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="add_darknetaa53_fpn_ul5_ep25_obj_focal", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 1, 'beta': 6, 'k_best': 20},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=25, min_delta=0.0003, score_threshold=0.05, nms_threshold=0.1)

num_epochs = 25

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### PAN Neck + WeightedNMS

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    neck_class=PAN,
    head_class=Head,
    unfreeze_last_backbone=5,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 1e-3},
    {'params': raw.neck.parameters(), 'lr': 1e-3},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=5e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)

reg_loss = diou_loss
obj_loss = FocalLoss()
cls_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(reg_loss, obj_loss, cls_loss, weight_bbox=7)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="darknetaa53_pan_ul5_ep30_obj_focal", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 1, 'beta': 6, 'k_best': 20},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=25, min_delta=0.0003, score_threshold=0.05, nms_threshold=0.1)

num_epochs = 30

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### Та же модель, но чуть правим параметры обучения

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 5e-4},
    {'params': raw.neck.parameters(), 'lr': 5e-4},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30, eta_min=1e-5)

reg_loss = diou_loss
obj_loss = FocalLoss()
cls_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(reg_loss, obj_loss, cls_loss, weight_bbox=7)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="add_darknetaa53_pan_ul5_ep30_obj_focal", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 6, 'beta': 1, 'k_best': 15},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=25, min_delta=0.0003, score_threshold=0.1, nms_threshold=0.5)

num_epochs = 30

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### Та же модель, добавим больше эмоъ и разморозим больше слоев

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    neck_class=PAN,
    head_class=Head,
    unfreeze_last_backbone=14,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)

RUN_ID = "5c89636df28f473aa80660e5557e264c"
ARTIFACT_NAME = "add_darknetaa53_pan_ul5_ep30_obj_focal_best.pt"
DOWNLOAD_DIR = "/kaggle/working/downloaded_models"

client = mlflow.tracking.MlflowClient()
local_weight_path = client.download_artifacts(
    run_id=RUN_ID,
    path=ARTIFACT_NAME,
    dst_path=DOWNLOAD_DIR
)

In [ ]:
state_dict = torch.load(local_weight_path, map_location=device, weights_only=False)
model.load_state_dict(state_dict["model_state_dict"])
model = model.to(device)

In [ ]:
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 3e-4},
    {'params': raw.neck.parameters(), 'lr': 3e-4},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=1e-5)

reg_loss = diou_loss
obj_loss = FocalLoss()
cls_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(reg_loss, obj_loss, cls_loss, weight_bbox=7)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="add_add_darknetaa53_pan_ul14_ep60_obj_focal", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 6, 'beta': 1, 'k_best': 15},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=60, min_delta=0.0003, score_threshold=0.1, nms_threshold=0.5)

num_epochs = 60

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### Та же модель, еще 40 эпох, разморозка еще слоев backbone

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    neck_class=PAN,
    head_class=Head,
    unfreeze_last_backbone=16,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)

RUN_ID = "e72872fe27804c019c1a1596d752e1cf"
ARTIFACT_NAME = "add_add_darknetaa53_pan_ul14_ep60_obj_focal_best.pt"
DOWNLOAD_DIR = "/kaggle/working/downloaded_models"

client = mlflow.tracking.MlflowClient()
local_weight_path = client.download_artifacts(
    run_id=RUN_ID,
    path=ARTIFACT_NAME,
    dst_path=DOWNLOAD_DIR
)

In [ ]:
state_dict = torch.load(local_weight_path, map_location=device, weights_only=False)
model.load_state_dict(state_dict["model_state_dict"])
model = model.to(device)

In [ ]:
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

In [ ]:
lr = 1e-3

raw = model.module if isinstance(model, nn.DataParallel) else model
optimizer = optim.AdamW(params=[
    {'params': raw.head.parameters(), 'lr': 3e-4},
    {'params': raw.neck.parameters(), 'lr': 3e-4},
    {'params': [p for p in raw.backbone.parameters() if p.requires_grad], 'lr': 1e-4}
], lr=lr, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=40, eta_min=1e-5)

reg_loss = diou_loss
obj_loss = FocalLoss()
cls_loss = nn.CrossEntropyLoss()
compute_loss = ComputeLoss(reg_loss, obj_loss, cls_loss, weight_bbox=7)

runner = Runner(model, compute_loss, optimizer, train_dataloader, TAL_assigner, 
                name_to_save="add_add_add_darknetaa53_pan_ul16_ep40_obj_focal", device=device,
                 scheduler=scheduler, assign_target_kwargs={'alpha': 6, 'beta': 1, 'k_best': 15},
                 val_dataloader=test_dataloader, checkpoint_dir='/kaggle/working/checkpoints',
                patience=60, min_delta=0.0003, score_threshold=0.1, nms_threshold=0.5)

num_epochs = 40

In [ ]:
runner.train(num_epochs=num_epochs, verbose=True)

### Пример загрузки весов обученной модели

In [ ]:
model = Detector(
    backbone_model_name='darknetaa53',
    unfreeze_last_backbone=15,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)

RUN_ID = "5285325cceaa4c79882868941e71c6b8"
ARTIFACT_NAME = "darknetaa53_fpn_ul15_ep50_no_early_stopping_best.pt"
DOWNLOAD_DIR = "/kaggle/working/downloaded_models"

client = mlflow.tracking.MlflowClient()
local_weight_path = client.download_artifacts(
    run_id=RUN_ID,
    path=ARTIFACT_NAME,
    dst_path=DOWNLOAD_DIR
)

In [ ]:
state_dict = torch.load(local_weight_path, map_location=device, weights_only=False)
model.load_state_dict(state_dict["model_state_dict"])
model = model.to(device)

In [ ]:
runner.nms_threshold=0.5
runner.score_threshold = 0.1
runner.validate(test_dataloader)

### Предикт с TTA

In [42]:
model = Detector(
    backbone_model_name='darknetaa53',
    neck_class=PAN,
    head_class=Head,
    unfreeze_last_backbone=16,
    out_indices_backbone=(-3, -2, -1),
    neck_n_channels=256,
    num_classes=4,
    anchor_sizes=((30, 46, 60), (66, 80, 96), (100, 116, 130)),
    anchor_ratios=((0.5, 1), (0.5, 1), (0.5, 1)),
    input_size=(640, 640),
).to(device)

RUN_ID = "43fe63d179df41dc9adbd87369c6e48c"
ARTIFACT_NAME = "add_add_add_darknetaa53_pan_ul16_ep40_obj_focal_best.pt"
DOWNLOAD_DIR = "/kaggle/working/downloaded_models"

client = mlflow.tracking.MlflowClient()
local_weight_path = client.download_artifacts(
    run_id=RUN_ID,
    path=ARTIFACT_NAME,
    dst_path=DOWNLOAD_DIR
)

In [43]:
state_dict = torch.load(local_weight_path, map_location=device, weights_only=False)
model.load_state_dict(state_dict["model_state_dict"])
model = model.to(device)

In [44]:
if torch.cuda.device_count() > 1:
    print("Trainig on 2 GPUs")
    model = nn.DataParallel(model)

Trainig on 2 GPUs


In [45]:
import torch
import torch.nn.functional as F
from torchvision.ops import box_iou

In [50]:
def _hflip_boxes(boxes, img_width):
    """ Отражает боксы (xyxy) по горизонтали обратно в исходную систему координат. """
    flipped = boxes.clone()
    flipped[:, 0] = img_width - boxes[:, 2]
    flipped[:, 2] = img_width - boxes[:, 0]
    return flipped


@torch.no_grad()
def tta_predict(model, image, device, scales=(640, 800), use_flip=True,
                 score_threshold=0.1, nms_threshold=0.5):
    """ Test-Time Augmentation для детектора.

    Параметры
    ---------
    model : Detector (в eval-режиме)
    image : Tensor [C, H, W], уже нормализованный, без batch-размерности
    device : устройство
    scales : на каких разрешениях прогонять модель (кроме нативного model.input_size)
    use_flip : добавлять ли horizontal flip к каждому масштабу
    score_threshold, nms_threshold : параметры финального объединения

    Returns
    -------
    dict с ключами 'boxes', 'labels', 'scores' — объединённые предсказания в системе
    координат исходного изображения.
    """
    model.eval()
    raw_model = model.module if isinstance(model, torch.nn.DataParallel) else model
    orig_h, orig_w = image.shape[-2:]

    all_boxes, all_scores, all_labels = [], [], []

    for scale in scales:
        # Ресайзим изображение под нужный масштаб (модель ожидает фиксированный input_size)
        img_scaled = F.interpolate(image.unsqueeze(0), size=(scale, scale),
                                     mode='bilinear', align_corners=False)
        scale_x = orig_w / scale
        scale_y = orig_h / scale

        transforms = [("orig", img_scaled)]
        if use_flip:
            transforms.append(("flip", torch.flip(img_scaled, dims=[-1])))

        for name, img_t in transforms:
            img_t = img_t.to(device)
            bboxes, confidences, cls_probs = model(img_t)  # [1, num_anchors, ...]

            bboxes = bboxes[0]
            confidences = confidences[0]
            cls_probs = cls_probs[0]

            # Возвращаем боксы в масштаб текущего прогона (модель работает в scale x scale)
            if name == "flip":
                bboxes = _hflip_boxes(bboxes, img_width=scale)

            # Пересчитываем координаты обратно в масштаб исходного изображения
            bboxes = bboxes.clone()
            bboxes[:, [0, 2]] *= scale_x
            bboxes[:, [1, 3]] *= scale_y

            final_scores = confidences.unsqueeze(-1) * cls_probs  # [num_anchors, num_classes]

            for cls in range(raw_model.num_classes):
                cls_scores = final_scores[:, cls]
                keep = cls_scores > score_threshold
                if keep.sum() == 0:
                    continue
                all_boxes.append(bboxes[keep])
                all_scores.append(cls_scores[keep])
                all_labels.append(torch.full((keep.sum(),), cls, device=device))

    if len(all_boxes) == 0:
        return {"boxes": torch.empty((0, 4)), "labels": torch.empty((0,), dtype=torch.long),
                "scores": torch.empty((0,))}

    all_boxes = torch.cat(all_boxes, dim=0)
    all_scores = torch.cat(all_scores, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    # Объединяем предсказания со всех прогонов отдельно по каждому классу через weighted NMS
    final_boxes, final_scores, final_labels = [], [], []
    for cls in range(raw_model.num_classes):
        cls_mask = all_labels == cls
        if cls_mask.sum() == 0:
            continue
        merged_boxes, merged_scores = weighted_nms(all_boxes[cls_mask], all_scores[cls_mask], nms_threshold)
        final_boxes.append(merged_boxes)
        final_scores.append(merged_scores)
        final_labels.append(torch.full((len(merged_boxes),), cls, device=device))

    return {
        "boxes": torch.cat(final_boxes, dim=0),
        "labels": torch.cat(final_labels, dim=0),
        "scores": torch.cat(final_scores, dim=0),
    }

In [51]:
@torch.no_grad()
def validate_with_tta(model, dataloader, device, num_classes, scales=(640,)):
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox")
    for images, targets in dataloader:
        preds = []
        for img in images:
            pred = tta_predict(model, img, device, scales=scales)
            # Переносим предсказания на CPU — targets из dataloader тоже на CPU
            pred = {k: v.cpu() for k, v in pred.items()}
            preds.append(pred)
        metric.update(preds, targets)
    return metric.compute()["map"].item()

In [52]:
validate_with_tta(model, test_dataloader, device, num_classes=4, scales=(640,))

0.20708920061588287

### Нарисуем предсказания модели

In [ ]:
import cv2

In [ ]:
# Определяем константы для цвета и названий классов
class_to_color = {
    1: (89, 161, 197),
    2: (204, 79, 135),
    3: (125, 216, 93),
    4: (175, 203, 33),
}

class_to_name = {
    1 : "enemy",
    2 : "enemy-head",
    3 : "friendly",
    4 : "friendly-head"
}

In [ ]:
def add_bbox(image, box, label='', color=(128, 128, 128), txt_color=(0, 0, 0)):
    lw = max(round(sum(image.shape) / 2 * 0.003), 2)
    p1, p2 = (int(box[0]), int(box[1])), (int(box[0]) + int(box[2]), int(box[1]) + int(box[3]))
    cv2.rectangle(image, p1, p2, color, thickness=lw, lineType=cv2.LINE_AA)
    if label:
        tf = max(lw - 1, 1)
        w, h = cv2.getTextSize(label, 0, fontScale=lw / 3, thickness=tf)[0]
        outside = p1[1] - h >= 3
        p2 = p1[0] + w, p1[1] - h - 3 if outside else p1[1] + h + 3
        cv2.rectangle(image, p1, p2, color, -1, cv2.LINE_AA)
        cv2.putText(image,
                    label, (p1[0], p1[1] - 2 if outside else p1[1] + h + 2),
                    0,
                    lw / 3,
                    txt_color,
                    thickness=tf,
                    lineType=cv2.LINE_AA)
    return image

In [ ]:
@torch.no_grad()
def predict(model, images, device, score_threshold=0.1, nms_threshold=0.5, max_boxes_per_cls=8, return_type='list'):
    """ Предсказание моделью для переданного набора изображений после фильтрации по score_threshold
    и применения NMS.

    Параметры
    --------
    images : torch.tensor, содержащий картинки для которых нужно сделать предсказание.
    Необходимые преобразования должны быть сделаны ДО. Внутри метода `predict` никаких преобразований
    не происходит.
    score_threshold : Все предсказания, с (confidence score * cls_probs) < score_threshold будут проигнорированны.
    nms_threshold : Предсказания, имеющие пересечение по IoU >= nms_threshold будут считаться одним предсказанием.
    max_boxes_per_cls : Максимальное количество ббоксов на изображение для одного класса после фильтрации по `score_threshold`.

    Returns
    -------
    final_predictions : List[dict], где каждый словарь содержащий следующие ключи:
        "boxes" : координаты ббоксов на i-ом изображении,
        "labels" : классы внутри ббоксов,
        "scores" : Confidence scores для ббоксов.
    """
    model.eval()
    images = images.to(device)
    outputs = model(images)
    final_predictions =  _filter_predictions(outputs, score_threshold=score_threshold, nms_threshold=nms_threshold,
                                             max_boxes_per_cls=max_boxes_per_cls, return_type=return_type)
    return final_predictions

In [ ]:
def plot_predictions(images, predictions, figsize=(12, 3)):
    """ Рисуем по 3 предсказания на одной строке. """
    ncols = min(len(images), 3)
    for ix in range(0, len(images), ncols):
        _, axes = plt.subplots(1, ncols, figsize=figsize, tight_layout=True)
        for i, (ax, img) in enumerate(zip(axes, images[ix: ix+ncols])):
            img = img.cpu().permute(1, 2, 0).numpy()
            img = img * np.array(std).reshape(1, 1, -1) + np.array(mean).reshape(1, 1, -1)
            img = np.ascontiguousarray((img * 255).astype(np.uint8))
            preds = predictions[ix + i]
            for bbox, label, score in zip(preds["boxes"], preds["labels"], preds["scores"]):
                color = class_to_color[label+1]
                label = class_to_name[label+1]
                img = add_bbox(img, bbox, label=f"Class {label}: {score:.2f}", color=color)
            ax.imshow(img)
            ax.set_xticks([])
            ax.set_yticks([])
        plt.show()
    plt.close()

In [ ]:
test_iter = iter(test_dataloader)

score_threshold = 0.1
nms_threshold = 0.1

images, _ = next(test_iter)
preds = predict(model, images, device=device, score_threshold=score_threshold, nms_threshold=nms_threshold)
plot_predictions(images, preds)